In [1]:
# ================================================================
# STAGE 12B - PAPER-ALIGNED CNN + LSTM
# ================================================================
#
# Base paper:
# "Spatio-temporal Learning for Trajectory Prediction
#  in Full Self-Driving Cars"
#
# Paper-described preprocessing:
#   1. Z-score based outlier removal
#   2. Min-Max normalization
#
# Paper architecture:
#
#        Input [B, 8, 5]
#              |
#           Conv1D
#        64 filters, k=3
#              |
#        [B, 6, 64]
#              |
#         MaxPool
#              |
#        [B, 3, 64]
#              |
#          Flatten
#              |
#        [B, 192]
#              |
#         Dense(64)
#              |
#         LSTM(64)
#              |
#       ┌──────┴──────┐
#       ↓             ↓
#   LinearX(12)   LinearY(12)
#       ↓             ↓
#      X(t)          Y(t)
#
# IMPORTANT:
# The paper does not specify the exact Z-score threshold or
# the exact Dense-to-LSTM reshaping implementation.
# Therefore those implementation details are explicitly
# configurable below.
#
# ================================================================


import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras import Model

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Reshape,
    LSTM,
    Concatenate
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from tensorflow.keras.metrics import RootMeanSquaredError


# ================================================================
# 1. CONFIGURATION
# ================================================================

BASE_DIR = r"C:\Users\natra\Final year project"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ================================================================
# 2. REPRODUCIBILITY
# ================================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)

np.random.seed(SEED)

tf.random.set_seed(SEED)


# ================================================================
# 3. TEMPORAL CONFIGURATION
# ================================================================

HISTORY_STEPS = 8

FUTURE_STEPS = 12

SAMPLING_FREQUENCY_HZ = 10

DT_SECONDS = 0.1

PREDICTION_HORIZON_SECONDS = FUTURE_STEPS * DT_SECONDS


# ================================================================
# 4. PAPER FEATURES
# ================================================================

# Our X_motion_10hz contains:
#
# 0 = relative_x
# 1 = relative_y
# 2 = vx
# 3 = vy
# 4 = speed
# 5 = acceleration
# 6 = heading_rate
# 7 = heading_sin
# 8 = heading_cos
#
# Paper uses:
#
# x, y, speed, acceleration, heading rate
#
# Therefore:

FEATURE_INDICES = [
    0,   # relative x
    1,   # relative y
    4,   # speed
    5,   # acceleration
    6    # heading rate
]

FEATURE_NAMES = [
    "x",
    "y",
    "speed",
    "acceleration",
    "heading_rate"
]

NUM_FEATURES = len(FEATURE_INDICES)


# ================================================================
# 5. TRAINING CONFIGURATION
# ================================================================

EPOCHS = 35

BATCH_SIZE = 128

LEARNING_RATE = 0.001

MISS_THRESHOLD_METERS = 2.0


# ================================================================
# 6. Z-SCORE CONFIGURATION
# ================================================================

# The paper states that Z-score based outlier removal was used,
# but does NOT specify the exact threshold.
#
# 3.0 is used here as a standard implementation choice.
#
# IMPORTANT:
# We use TRAINING statistics only.
#
# We do NOT use test-set statistics.
#
# We also keep the test set unchanged so that the evaluation
# remains honest and reproducible.

Z_SCORE_THRESHOLD = 3.0


# ================================================================
# 7. PRINT CONFIGURATION
# ================================================================

print("=" * 80)

print("STAGE 12B - PAPER-ALIGNED CNN + LSTM")

print("=" * 80)

print()

print("Input:")
print(
    f"({HISTORY_STEPS}, {NUM_FEATURES})"
)

print()

print("Features:")
print(FEATURE_NAMES)

print()

print(
    f"History observations : {HISTORY_STEPS}"
)

print(
    f"Sampling frequency    : {SAMPLING_FREQUENCY_HZ} Hz"
)

print(
    f"Time interval         : {DT_SECONDS} sec"
)

print(
    f"Future points         : {FUTURE_STEPS}"
)

print(
    f"Prediction horizon    : "
    f"{PREDICTION_HORIZON_SECONDS:.1f} sec"
)

print()

print(
    f"Z-score threshold     : ±{Z_SCORE_THRESHOLD}"
)

print("=" * 80)


# ================================================================
# 8. LOAD DATA
# ================================================================

print("\n" + "=" * 80)

print("LOADING 10-HZ DATA")

print("=" * 80)


def load_split(split):

    motion_path = os.path.join(
        OUTPUT_DIR,
        f"X_motion_10hz_{split}.npy"
    )

    target_path = os.path.join(
        OUTPUT_DIR,
        f"Y_10hz_{split}.npy"
    )

    if not os.path.exists(motion_path):

        raise FileNotFoundError(
            f"Missing file:\n{motion_path}"
        )

    if not os.path.exists(target_path):

        raise FileNotFoundError(
            f"Missing file:\n{target_path}"
        )

    motion = np.load(
        motion_path
    )

    target = np.load(
        target_path
    )

    return motion, target


X_train_full, Y_train = load_split("train")

X_val_full, Y_val = load_split("val")

X_test_full, Y_test = load_split("test")


# ================================================================
# 9. SELECT PAPER FEATURES
# ================================================================

X_train = X_train_full[
    :,
    :,
    FEATURE_INDICES
]

X_val = X_val_full[
    :,
    :,
    FEATURE_INDICES
]

X_test = X_test_full[
    :,
    :,
    FEATURE_INDICES
]


print("\nTRAIN")

print(
    "X:",
    X_train.shape
)

print(
    "Y:",
    Y_train.shape
)


print("\nVAL")

print(
    "X:",
    X_val.shape
)

print(
    "Y:",
    Y_val.shape
)


print("\nTEST")

print(
    "X:",
    X_test.shape
)

print(
    "Y:",
    Y_test.shape
)


# ================================================================
# 10. DATA QUALITY CHECK
# ================================================================

print("\n" + "=" * 80)

print("DATA QUALITY CHECK")

print("=" * 80)


def check_array(name, array):

    nan_count = np.isnan(array).sum()

    inf_count = np.isinf(array).sum()

    print(
        f"{name:12s} "
        f"NaN={nan_count} "
        f"Inf={inf_count}"
    )

    if nan_count > 0:

        raise ValueError(
            f"{name} contains NaN values."
        )

    if inf_count > 0:

        raise ValueError(
            f"{name} contains Inf values."
        )


check_array("X_train", X_train)

check_array("Y_train", Y_train)

check_array("X_val", X_val)

check_array("Y_val", Y_val)

check_array("X_test", X_test)

check_array("Y_test", Y_test)


# ================================================================
# 11. Z-SCORE OUTLIER REMOVAL
# ================================================================
#
# We calculate statistics using TRAINING data only.
#
# For each feature:
#
#        z = (x - mean) / std
#
# Samples containing a value outside ±3 standard deviations
# are considered outliers.
#
# We remove those samples from TRAINING DATA ONLY.
#
# Validation and test datasets are NOT filtered.
#
# This avoids using test information during preprocessing.
#
# ================================================================

print("\n" + "=" * 80)

print("Z-SCORE OUTLIER REMOVAL")

print("=" * 80)


# ------------------------------------------------
# Calculate training statistics
# ------------------------------------------------

train_motion_flat = X_train.reshape(
    -1,
    NUM_FEATURES
)

motion_mean = np.mean(
    train_motion_flat,
    axis=0
)

motion_std = np.std(
    train_motion_flat,
    axis=0
)

# Avoid division by zero

motion_std = np.where(
    motion_std < 1e-8,
    1.0,
    motion_std
)


# ------------------------------------------------
# Calculate Z-score for every training point
# ------------------------------------------------

motion_z = (
    train_motion_flat - motion_mean
) / motion_std


motion_z = motion_z.reshape(
    X_train.shape
)


# ------------------------------------------------
# Determine which sequences are valid
# ------------------------------------------------

motion_valid = np.all(
    np.abs(motion_z) <= Z_SCORE_THRESHOLD,
    axis=(1, 2)
)


# ------------------------------------------------
# Also check target trajectory
# ------------------------------------------------

target_flat = Y_train.reshape(
    -1,
    2
)

target_mean = np.mean(
    target_flat,
    axis=0
)

target_std = np.std(
    target_flat,
    axis=0
)

target_std = np.where(
    target_std < 1e-8,
    1.0,
    target_std
)


target_z = (
    target_flat - target_mean
) / target_std


target_z = target_z.reshape(
    Y_train.shape
)


target_valid = np.all(
    np.abs(target_z) <= Z_SCORE_THRESHOLD,
    axis=(1, 2)
)


# ------------------------------------------------
# Final training mask
# ------------------------------------------------

train_valid_mask = (
    motion_valid &
    target_valid
)


original_train_count = len(
    X_train
)

filtered_train_count = np.sum(
    train_valid_mask
)

removed_train_count = (
    original_train_count -
    filtered_train_count
)


print()

print(
    "Original training sequences:",
    original_train_count
)

print(
    "Remaining training sequences:",
    filtered_train_count
)

print(
    "Removed training sequences:",
    removed_train_count
)

print(
    "Removal percentage:",
    f"{100 * removed_train_count / original_train_count:.2f}%"
)


# ================================================================
# APPLY FILTER
# ================================================================

X_train_filtered = X_train[
    train_valid_mask
]

Y_train_filtered = Y_train[
    train_valid_mask
]


print()

print(
    "Filtered X_train:",
    X_train_filtered.shape
)

print(
    "Filtered Y_train:",
    Y_train_filtered.shape
)


# ================================================================
# IMPORTANT:
# KEEP VALIDATION AND TEST UNCHANGED
# ================================================================

print()

print(
    "Validation set: unchanged"
)

print(
    "Test set      : unchanged"
)


# ================================================================
# 12. MIN-MAX NORMALIZATION
# ================================================================

print("\n" + "=" * 80)

print("MIN-MAX NORMALIZATION")

print("=" * 80)


motion_scaler = MinMaxScaler()


motion_scaler.fit(
    X_train_filtered.reshape(
        -1,
        NUM_FEATURES
    )
)


# ------------------------------------------------
# Train
# ------------------------------------------------

X_train_scaled = motion_scaler.transform(
    X_train_filtered.reshape(
        -1,
        NUM_FEATURES
    )
).reshape(
    X_train_filtered.shape
).astype(
    np.float32
)


# ------------------------------------------------
# Validation
# ------------------------------------------------

X_val_scaled = motion_scaler.transform(
    X_val.reshape(
        -1,
        NUM_FEATURES
    )
).reshape(
    X_val.shape
).astype(
    np.float32
)


# ------------------------------------------------
# Test
# ------------------------------------------------

X_test_scaled = motion_scaler.transform(
    X_test.reshape(
        -1,
        NUM_FEATURES
    )
).reshape(
    X_test.shape
).astype(
    np.float32
)


# ================================================================
# 13. TARGET MIN-MAX NORMALIZATION
# ================================================================

target_scaler = MinMaxScaler()


target_scaler.fit(
    Y_train_filtered.reshape(
        -1,
        2
    )
)


Y_train_scaled = target_scaler.transform(
    Y_train_filtered.reshape(
        -1,
        2
    )
).reshape(
    Y_train_filtered.shape
).astype(
    np.float32
)


Y_val_scaled = target_scaler.transform(
    Y_val.reshape(
        -1,
        2
    )
).reshape(
    Y_val.shape
).astype(
    np.float32
)


Y_test_scaled = target_scaler.transform(
    Y_test.reshape(
        -1,
        2
    )
).reshape(
    Y_test.shape
).astype(
    np.float32
)


print(
    "Min-Max normalization complete."
)


# ================================================================
# 14. SAVE NORMALIZATION INFORMATION
# ================================================================

normalization_path = os.path.join(
    OUTPUT_DIR,
    "stage12b_paper_aligned_normalization.npz"
)


np.savez(
    normalization_path,

    motion_data_min=motion_scaler.data_min_,

    motion_data_max=motion_scaler.data_max_,

    motion_scale=motion_scaler.scale_,

    motion_min=motion_scaler.min_,

    target_data_min=target_scaler.data_min_,

    target_data_max=target_scaler.data_max_,

    target_scale=target_scaler.scale_,

    target_min=target_scaler.min_,

    z_score_threshold=Z_SCORE_THRESHOLD
)


print(
    "Saved:",
    normalization_path
)


# ================================================================
# 15. BUILD PAPER-ALIGNED MODEL
# ================================================================

print("\n" + "=" * 80)

print("BUILDING MODEL")

print("=" * 80)


# ------------------------------------------------
# INPUT
# ------------------------------------------------

inputs = Input(
    shape=(
        HISTORY_STEPS,
        NUM_FEATURES
    ),
    name="paper_input"
)


# ------------------------------------------------
# CNN
# ------------------------------------------------

x = Conv1D(
    filters=64,
    kernel_size=3,
    padding="valid",
    activation="relu",
    name="Conv1D_64"
)(inputs)


print(
    "After Conv1D:",
    x.shape
)


# Expected:
#
# (None, 6, 64)


# ------------------------------------------------
# MAX POOL
# ------------------------------------------------

x = MaxPooling1D(
    pool_size=2,
    strides=2,
    padding="valid",
    name="MaxPool"
)(x)


print(
    "After MaxPool:",
    x.shape
)


# Expected:
#
# (None, 3, 64)


# ------------------------------------------------
# FLATTEN
# ------------------------------------------------

x = Flatten(
    name="Flatten"
)(x)


print(
    "After Flatten:",
    x.shape
)


# Expected:
#
# (None, 192)


# ------------------------------------------------
# DENSE
# ------------------------------------------------

x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)


print(
    "After Dense:",
    x.shape
)


# Expected:
#
# (None, 64)


# ================================================================
# DENSE → LSTM
# ================================================================
#
# The paper figure does not specify the exact reshape.
#
# We therefore preserve the 64-dimensional representation and
# interpret it as a one-step sequence.
#
# ================================================================

x = Reshape(
    (1, 64),
    name="LSTM_sequence"
)(x)


print(
    "Before LSTM:",
    x.shape
)


# ------------------------------------------------
# LSTM
# ------------------------------------------------

x = LSTM(
    64,
    return_sequences=False,
    name="LSTM_64"
)(x)


print(
    "After LSTM:",
    x.shape
)


# ================================================================
# SEPARATE X DECODER
# ================================================================

out_x = Dense(
    FUTURE_STEPS,
    activation="linear",
    name="LinearX"
)(x)


# ================================================================
# SEPARATE Y DECODER
# ================================================================

out_y = Dense(
    FUTURE_STEPS,
    activation="linear",
    name="LinearY"
)(x)


# ================================================================
# RESHAPE X
# ================================================================

out_x = Reshape(
    (
        FUTURE_STEPS,
        1
    ),
    name="X_trajectory"
)(out_x)


# ================================================================
# RESHAPE Y
# ================================================================

out_y = Reshape(
    (
        FUTURE_STEPS,
        1
    ),
    name="Y_trajectory"
)(out_y)


# ================================================================
# COMBINE X + Y
# ================================================================

outputs = Concatenate(
    axis=-1,
    name="future_trajectory"
)(
    [
        out_x,
        out_y
    ]
)


# ================================================================
# CREATE MODEL
# ================================================================

model = Model(
    inputs=inputs,
    outputs=outputs,
    name="Paper_Aligned_CNN_LSTM"
)


# ================================================================
# 16. COMPILE
# ================================================================

model.compile(
    optimizer=Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="mse",
    metrics=[
        RootMeanSquaredError(
            name="rmse"
        )
    ]
)


# ================================================================
# MODEL SUMMARY
# ================================================================

print()

model.summary()


# ================================================================
# 17. CALLBACKS
# ================================================================

model_path = os.path.join(
    OUTPUT_DIR,
    "stage12b_paper_aligned_cnn_lstm_best.keras"
)


callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    ModelCheckpoint(
        model_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


# ================================================================
# 18. TRAIN
# ================================================================

print("\n" + "=" * 80)

print("TRAINING")

print("=" * 80)


history = model.fit(

    X_train_scaled,

    Y_train_scaled,

    validation_data=(

        X_val_scaled,

        Y_val_scaled

    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=callbacks,

    verbose=1
)


# ================================================================
# 19. SAVE TRAINING HISTORY
# ================================================================

history_path = os.path.join(
    OUTPUT_DIR,
    "stage12b_paper_aligned_training.csv"
)


history_df = pd.DataFrame(
    history.history
)


history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(history_df) + 1
    )
)


history_df.to_csv(
    history_path,
    index=False
)


print(
    "Saved:",
    history_path
)


# ================================================================
# 20. TRAINING CURVE
# ================================================================

curve_path = os.path.join(
    OUTPUT_DIR,
    "stage12b_paper_aligned_training_curve.png"
)


plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "MSE Loss"
)

plt.title(
    "Paper-Aligned CNN + LSTM Training"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    curve_path,
    dpi=200
)

plt.close()


# ================================================================
# 21. LOAD BEST MODEL
# ================================================================

print("\n" + "=" * 80)

print("LOADING BEST MODEL")

print("=" * 80)


model = tf.keras.models.load_model(
    model_path
)


print(
    "Best model loaded."
)


# ================================================================
# 22. TEST PREDICTION
# ================================================================

print("\n" + "=" * 80)

print("GENERATING TEST PREDICTIONS")

print("=" * 80)


Y_pred_scaled = model.predict(
    X_test_scaled,
    batch_size=BATCH_SIZE,
    verbose=1
)


print(
    "Prediction shape:",
    Y_pred_scaled.shape
)


# ================================================================
# 23. INVERSE TRANSFORM
# ================================================================

Y_pred = target_scaler.inverse_transform(

    Y_pred_scaled.reshape(
        -1,
        2
    )

).reshape(
    Y_pred_scaled.shape
)


# ================================================================
# 24. SAVE PREDICTIONS
# ================================================================

prediction_path = os.path.join(
    OUTPUT_DIR,
    "stage12b_paper_aligned_predictions.npy"
)


np.save(
    prediction_path,
    Y_pred
)


print(
    "Saved:",
    prediction_path
)


# ================================================================
# 25. CALCULATE POINT ERRORS
# ================================================================

point_errors = np.sqrt(

    np.sum(

        (
            Y_pred -
            Y_test
        ) ** 2,

        axis=-1

    )
)


# ================================================================
# 26. ADE
# ================================================================

sample_ADE = np.mean(
    point_errors,
    axis=1
)


FINAL_ADE = np.mean(
    sample_ADE
)


# ================================================================
# 27. FDE
# ================================================================

sample_FDE = point_errors[
    :,
    -1
]


FINAL_FDE = np.mean(
    sample_FDE
)


# ================================================================
# 28. RMSE
# ================================================================

FINAL_RMSE = np.sqrt(

    np.mean(

        (
            Y_pred -
            Y_test
        ) ** 2

    )
)


# ================================================================
# 29. MISS RATE
# ================================================================

MISS_RATE = (

    np.mean(

        sample_FDE >
        MISS_THRESHOLD_METERS

    )

    * 100

)


# ================================================================
# 30. PRINT RESULTS
# ================================================================

print("\n" + "=" * 80)

print("PAPER-ALIGNED CNN + LSTM RESULTS")

print("=" * 80)

print()

print(
    f"ADE  = {FINAL_ADE:.4f} m"
)

print(
    f"FDE  = {FINAL_FDE:.4f} m"
)

print(
    f"RMSE = {FINAL_RMSE:.4f} m"
)

print(
    f"MR   = {MISS_RATE:.2f}%"
)

print()

print(
    f"Test sequences = {len(Y_test)}"
)

print(
    f"Training sequences after Z-score filtering = "
    f"{len(X_train_filtered)}"
)

print(
    f"Sampling frequency = "
    f"{SAMPLING_FREQUENCY_HZ} Hz"
)

print(
    f"Prediction horizon = "
    f"{PREDICTION_HORIZON_SECONDS:.1f} sec"
)

print()

print("=" * 80)


# ================================================================
# 32. SAVE RESULTS
# ================================================================

results_path = os.path.join(
    OUTPUT_DIR,
    "stage12b_paper_aligned_results.csv"
)


results_df = pd.DataFrame({

    "model": [
        "Paper-Aligned CNN + LSTM LinearX LinearY"
    ],

    "ADE_m": [
        FINAL_ADE
    ],

    "FDE_m": [
        FINAL_FDE
    ],

    "RMSE_m": [
        FINAL_RMSE
    ],

    "MR_percent": [
        MISS_RATE
    ],

    "test_sequences": [
        len(Y_test)
    ],

    "original_train_sequences": [
        original_train_count
    ],

    "filtered_train_sequences": [
        filtered_train_count
    ],

    "z_score_threshold": [
        Z_SCORE_THRESHOLD
    ],

    "history_steps": [
        HISTORY_STEPS
    ],

    "future_steps": [
        FUTURE_STEPS
    ],

    "sampling_frequency_hz": [
        SAMPLING_FREQUENCY_HZ
    ],

    "dt_seconds": [
        DT_SECONDS
    ],

    "prediction_horizon_sec": [
        PREDICTION_HORIZON_SECONDS
    ],

    "features": [
        "x,y,speed,acceleration,heading_rate"
    ]

})


results_df.to_csv(
    results_path,
    index=False
)


print(
    "Results saved:",
    results_path
)


# ================================================================
# 33. REPRESENTATIVE TRAJECTORY
# ================================================================

representative_index = int(

    np.argsort(
        sample_ADE
    )[

        len(sample_ADE) // 2

    ]

)


gt = Y_test[
    representative_index
]


pred = Y_pred[
    representative_index
]


history_xy = X_test[
    representative_index,
    :,
    :2
].copy()


# Convert history to relative coordinates

history_xy = (

    history_xy -
    history_xy[-1]

)


representative_ADE = sample_ADE[
    representative_index
]


representative_FDE = sample_FDE[
    representative_index
]


print()

print(
    "Representative sample:"
)

print(
    "Index:",
    representative_index
)

print(
    f"ADE: {representative_ADE:.4f} m"
)

print(
    f"FDE: {representative_FDE:.4f} m"
)


# ================================================================
# 34. TRAJECTORY PLOT
# ================================================================

trajectory_path = os.path.join(
    OUTPUT_DIR,
    "stage12b_paper_aligned_trajectory.png"
)


plt.figure(
    figsize=(8, 6)
)


plt.plot(
    history_xy[:, 0],
    history_xy[:, 1],
    marker="o",
    label="History"
)


plt.plot(
    gt[:, 0],
    gt[:, 1],
    marker="o",
    label="Ground Truth"
)


plt.plot(
    pred[:, 0],
    pred[:, 1],
    marker="x",
    label="Paper-Aligned Prediction"
)


plt.xlabel(
    "Relative X (m)"
)

plt.ylabel(
    "Relative Y (m)"
)

plt.title(
    "Paper-Aligned CNN + LSTM Trajectory"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.axis(
    "equal"
)

plt.tight_layout()


plt.savefig(
    trajectory_path,
    dpi=200
)


plt.close()


# ================================================================
# 35. FINAL
# ================================================================

print("\n" + "=" * 80)

print("STAGE 12B COMPLETE")

print("=" * 80)

print()

print("FINAL RESULT:")

print(
    f"ADE  = {FINAL_ADE:.4f} m"
)

print(
    f"FDE  = {FINAL_FDE:.4f} m"
)

print(
    f"RMSE = {FINAL_RMSE:.4f} m"
)

print(
    f"MR   = {MISS_RATE:.2f}%"
)

print(
    f"Test = {len(Y_test)} sequences"
)

print("=" * 80)

STAGE 12B - PAPER-ALIGNED CNN + LSTM

Input:
(8, 5)

Features:
['x', 'y', 'speed', 'acceleration', 'heading_rate']

History observations : 8
Sampling frequency    : 10 Hz
Time interval         : 0.1 sec
Future points         : 12
Prediction horizon    : 1.2 sec

Z-score threshold     : ±3.0

LOADING 10-HZ DATA

TRAIN
X: (352120, 8, 5)
Y: (352120, 12, 2)

VAL
X: (87707, 8, 5)
Y: (87707, 12, 2)

TEST
X: (84570, 8, 5)
Y: (84570, 12, 2)

DATA QUALITY CHECK
X_train      NaN=0 Inf=0
Y_train      NaN=0 Inf=0
X_val        NaN=0 Inf=0
Y_val        NaN=0 Inf=0
X_test       NaN=0 Inf=0
Y_test       NaN=0 Inf=0

Z-SCORE OUTLIER REMOVAL

Original training sequences: 352120
Remaining training sequences: 287661
Removed training sequences: 64459
Removal percentage: 18.31%

Filtered X_train: (287661, 8, 5)
Filtered Y_train: (287661, 12, 2)

Validation set: unchanged
Test set      : unchanged

MIN-MAX NORMALIZATION
Min-Max normalization complete.
Saved: C:\Users\natra\Final year project\outputs\stage12b

Model: "Paper_Aligned_CNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ paper_input (InputLayer)      │ (None, 8, 5)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1D_64 (Conv1D)            │ (None, 6, 64)             │           1,024 │ paper_input[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ MaxPool (MaxPooling1D)        │ (None, 3, 64)             │               0 │ Conv1D_64[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Flatten (Flatten)             │ (None, 192)               │               0 │ MaxPool[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Dense_64 (Dense)              │ (None, 64)                │          12,352 │ Flatten[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ LSTM_sequence (Reshape)       │ (None, 1, 64)             │               0 │ Dense_64[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ LSTM_64 (LSTM)                │ (None, 64)                │          33,024 │ LSTM_sequence[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ LinearX (Dense)               │ (None, 12)                │             780 │ LSTM_64[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ LinearY (Dense)               │ (None, 12)                │             780 │ LSTM_64[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ X_trajectory (Reshape)        │ (None, 12, 1)             │               0 │ LinearX[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Y_trajectory (Reshape)        │ (None, 12, 1)             │               0 │ LinearY[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ future_trajectory             │ (None, 12, 2)             │               0 │ X_trajectory[0][0],        │
│ (Concatenate)                 │                           │                 │ Y_trajectory[0][0]         │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 47,960 (187.34 KB)

 Trainable params: 47,960 (187.34 KB)

 Non-trainable params: 0 (0.00 B)


TRAINING
Epoch 1/35
2248/2248 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0023 - rmse: 0.0478
Epoch 1: val_loss improved from None to 0.00232, saving model to C:\Users\natra\Final year project\outputs\stage12b_paper_aligned_cnn_lstm_best.keras

Epoch 1: finished saving model to C:\Users\natra\Final year project\outputs\stage12b_paper_aligned_cnn_lstm_best.keras
2248/2248 ━━━━━━━━━━━━━━━━━━━━ 34s 13ms/step - loss: 0.0023 - rmse: 0.0478 - val_loss: 0.0023 - val_rmse: 0.0482 - learning_rate: 0.0010
Epoch 2/35
2244/2248 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 8.4133e-05 - rmse: 0.0092
Epoch 2: val_loss did not improve from 0.00232
2248/2248 ━━━━━━━━━━━━━━━━━━━━ 37s 11ms/step - loss: 8.4095e-05 - rmse: 0.0092 - val_loss: 0.0025 - val_rmse: 0.0499 - learning_rate: 0.0010
Epoch 3/35
2246/2248 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 7.9787e-05 - rmse: 0.0089
Epoch 3: val_loss did not improve from 0.00232
2248/2248 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - loss: 7.9780e-05 - rmse: 0.0089 - val_lo